# Groq Multi-Agent System

Run from the project root with the virtualenv active (`pip install -r requirements.txt && pip install -e .`).
Set `GROQ_API_KEY` in `.env` (see `.env.example`).

In [ ]:
import os
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

from dotenv import load_dotenv

load_dotenv(ROOT / ".env")
assert os.getenv("GROQ_API_KEY"), "Set GROQ_API_KEY in .env"
print("GROQ model:", os.getenv("GROQ_MODEL", "openai/gpt-oss-120b"))

In [ ]:
from langchain_core.messages import HumanMessage

from groq_multi_agent.graphs.supervisor import build_supervisor_graph

TASK = (
    "Research the tradeoffs of using Groq for a startup's customer-support "
    "assistant and write a one-page recommendation."
)

graph = build_supervisor_graph()
config = getattr(graph, "_groq_multi_agent_config", {"recursion_limit": 15})
result = graph.invoke(
    {"messages": [HumanMessage(content=TASK)], "next": "researcher"},
    config=config,
)

for msg in result["messages"]:
    name = getattr(msg, "name", None) or msg.type
    print(f"\n=== {name} ===\n{msg.content[:2000]}")

In [ ]:
from groq_multi_agent.graphs.hierarchy import build_hierarchy_graph

hgraph = build_hierarchy_graph()
hconfig = getattr(hgraph, "_groq_multi_agent_config", {"recursion_limit": 20})
hresult = hgraph.invoke(
    {"messages": [HumanMessage(content=TASK)], "next": "research_team"},
    config=hconfig,
)

for msg in hresult["messages"]:
    name = getattr(msg, "name", None) or msg.type
    print(f"\n=== {name} ===\n{msg.content[:2000]}")